In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Loading Airbnb DataSet
airbnb_path = "C:/Users/2058087/Desktop/DATASCI207/207 Project/LA_listings.csv"
airbnb = pd.read_csv(airbnb_path)
print("Raw Airbnb shape:", airbnb.shape) _
airbnb.head()

Raw Airbnb shape: (45533, 26)


C:\Users\2058087\AppData\Local\Temp\ipykernel_14736\4057463430.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  airbnb = pd.read_csv(airbnb_path)


,Unnamed: 0,id,name,host_id,host_name,host_since,host_response_time,host_response_rate,host_is_superhost,neighbourhood_cleansed,...,bathrooms,bedrooms,beds,price,minimum_nights,availability_365,number_of_reviews,review_scores_rating,license,instant_bookable
0,0,670339032744709144,Westwood lovely three bedrooms three bathrooms,4780152,Moon,20/01/13,within a few hours,0.96,f,West Los Angeles,...,3.0,3.0,3.0,399.0,30,365,0,NaN,NaN,f
1,1,37014494,Spanish style lower duplex near Beverly Hills,278288178,Ida,22/07/19,NaN,NaN,f,Beverlywood,...,NaN,2.0,NaN,NaN,30,0,0,NaN,NaN,f
2,2,1024835174766068422,Charming Beverly Hills Home,513813179,Tiana,08/05/23,within a day,0.60,f,Beverly Hills,...,3.0,3.0,3.0,434.0,30,267,0,NaN,NaN,f
3,3,850744632375448560,Tianpu's warm room with bathroom,432956623,Dan,22/11/21,a few days or more,0.20,f,Temple City,...,1.0,1.0,1.0,49.0,1,364,1,3.00,NaN,f
4,4,953950676345326970,"Santa Monica apt, free parking, steps to the b...",528669205,Farkhat,29/07/23,within an hour,1.00,t,Santa Monica,...,1.0,0.0,1.0,231.0,5,193,44,4.93,Exempt,t


In [ ]:
# Loading Zillow DataSet
zillow_path = "/Users/kalafejzo/Desktop/neighborhood_zillow.csv"
zillow = pd.read_csv(zillow_path)

print("Raw Zillow shape:", zillow.shape)

zillow.head()

Raw Zillow shape: (21600, 326)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2000-01-31,...,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31
0,112345,0,Maryvale,neighborhood,AZ,AZ,Phoenix,"Phoenix-Mesa-Chandler, AZ",Maricopa County,70265.246748,...,3.348063e+05,3.334858e+05,3.323610e+05,3.319901e+05,3.322638e+05,3.325127e+05,3.327364e+05,3.325103e+05,3.319002e+05,3.306516e+05
1,192689,1,Paradise,neighborhood,NV,NV,Las Vegas,"Las Vegas-Henderson-Paradise, NV",Clark County,136526.801960,...,4.010091e+05,3.995406e+05,3.983678e+05,3.977305e+05,3.972949e+05,3.969128e+05,3.965360e+05,3.960044e+05,3.954638e+05,3.943812e+05
2,270958,2,Upper West Side,neighborhood,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",New York County,409392.009209,...,1.316164e+06,1.314903e+06,1.318541e+06,1.324716e+06,1.329700e+06,1.337544e+06,1.347921e+06,1.355638e+06,1.357036e+06,1.353330e+06
3,270957,3,Upper East Side,neighborhood,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",New York County,666524.848137,...,1.272871e+06,1.278652e+06,1.287203e+06,1.294640e+06,1.298567e+06,1.307315e+06,1.323102e+06,1.337632e+06,1.343938e+06,1.339842e+06
4,118208,4,South Los Angeles,neighborhood,CA,CA,Los Angeles,"Los Angeles-Long Beach-Anaheim, CA",Los Angeles County,134453.790051,...,6.770960e+05,6.801553e+05,6.823887e+05,6.835706e+05,6.853429e+05,6.855409e+05,6.846294e+05,6.807705e+05,6.759697e+05,6.725631e+05


In [ ]:
# Clean Airbnb price column

airbnb = airbnb.copy()

# Convert price to numeric
airbnb["price"] = (
    airbnb["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
airbnb['price'] = pd.to_numeric(airbnb['price'], errors="coerce")

# Drop rows missing price because price is needed for target
airbnb = airbnb.dropna(subset=["price"])

# Remove zero or negative prices
airbnb = airbnb[airbnb["price"] > 0]

print("After price cleaning:", airbnb.shape)
airbnb.head()

After price cleaning: (37296, 26)


,Unnamed: 0,id,name,host_id,host_name,host_since,host_response_time,host_response_rate,host_is_superhost,neighbourhood_cleansed,...,bathrooms,bedrooms,beds,price,minimum_nights,availability_365,number_of_reviews,review_scores_rating,license,instant_bookable
0,0,670339032744709144,Westwood lovely three bedrooms three bathrooms,4780152,Moon,20/01/13,within a few hours,0.96,f,West Los Angeles,...,3.0,3.0,3.0,399.0,30,365,0,NaN,NaN,f
2,2,1024835174766068422,Charming Beverly Hills Home,513813179,Tiana,08/05/23,within a day,0.60,f,Beverly Hills,...,3.0,3.0,3.0,434.0,30,267,0,NaN,NaN,f
3,3,850744632375448560,Tianpu's warm room with bathroom,432956623,Dan,22/11/21,a few days or more,0.20,f,Temple City,...,1.0,1.0,1.0,49.0,1,364,1,3.00,NaN,f
4,4,953950676345326970,"Santa Monica apt, free parking, steps to the b...",528669205,Farkhat,29/07/23,within an hour,1.00,t,Santa Monica,...,1.0,0.0,1.0,231.0,5,193,44,4.93,Exempt,t
5,5,1082614257267236225,Practical Private Room #1398 A,557999481,Christos,22/01/24,within a day,1.00,f,North Hollywood,...,0.0,4.0,1.0,62.0,30,278,0,NaN,NaN,f


In [7]:
#Cleaning rest of columns
numeric_cols = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "bedrooms",
    "bathrooms"
]

for col in numeric_cols:
    airbnb[col] = pd.to_numeric(airbnb[col], errors="coerce")

# If reviews_per_month is missing, assume 0 reviews per month
airbnb["reviews_per_month"] = airbnb["reviews_per_month"].fillna(0)

print("After numeric cleaning:", airbnb.shape)
airbnb.head()

KeyError: 'reviews_per_month'

Our target column is the estimated annual revenue column

In [ ]:
#Create estimated annual revenue
airbnb["estimated_booked_nights"] = 365 - airbnb["availability_365"]

airbnb = airbnb[
    (airbnb["estimated_booked_nights"] >= 0) &
    (airbnb["estimated_booked_nights"] <= 365)
]

airbnb["estimated_annual_revenue"] = (
    airbnb["price"] * airbnb["estimated_booked_nights"]
)

print("After target creation:", airbnb.shape)

airbnb[["price", "availability_365", "estimated_booked_nights", "estimated_annual_revenue"]].head()

After target creation: (36819, 81)


,price,availability_365,estimated_booked_nights,estimated_annual_revenue
0,81.0,361,4,324.0
1,110.0,274,91,10010.0
4,88.0,291,74,6512.0
5,1000.0,182,183,183000.0
6,81.0,311,54,4374.0


We removed outliers because Airbnb price and estimated revenue are highly right-skewed and can contain extreme listings, such as luxury properties or unusual rentals, that are not representative of typical Airbnb activity. Since estimated annual revenue is calculated from price, extreme price values can create even more extreme revenue values and disproportionately influence the model. Filtering observations outside the 1st and 99th percentiles preserves most of the data while reducing the impact of unusually high or low values. We also removed listings with minimum-night requirements above 365 because they are not realistic for estimating annual short-term rental revenue.

In [ ]:
before_outliers = airbnb.shape[0]

price_low, price_high = airbnb["price"].quantile([0.01, 0.99])
revenue_low, revenue_high = airbnb["estimated_annual_revenue"].quantile([0.01, 0.99])

airbnb = airbnb[
    (airbnb["price"] >= price_low) &
    (airbnb["price"] <= price_high) &
    (airbnb["estimated_annual_revenue"] >= revenue_low) &
    (airbnb["estimated_annual_revenue"] <= revenue_high) &
    (airbnb["minimum_nights"] <= 365)
]

print("Rows before outlier filtering:", before_outliers)
print("Rows after outlier filtering:", airbnb.shape[0])
print("Rows removed:", before_outliers - airbnb.shape[0])

Rows before outlier filtering: 36819
Rows after outlier filtering: 35870
Rows removed: 949


In [ ]:
#Prepping Zillow Data

# Keep only California rows
zillow = zillow[zillow["State"] == "CA"]

# Find date columns and use most recent Zillow home value
date_cols = [col for col in zillow.columns if col[:4].isdigit()]
latest_date = sorted(date_cols)[-1]

zillow = zillow[["RegionName", latest_date]].copy()

zillow = zillow.rename(columns={
    "RegionName": "neighbourhood",
    latest_date: "latest_home_value"
})

zillow["neighbourhood_clean"] = (
    zillow["neighbourhood"]
    .astype(str)
    .str.lower()
    .str.strip()
)

zillow["latest_home_value"] = pd.to_numeric(
    zillow["latest_home_value"],
    errors="coerce"
)

print("Latest Zillow date used:", latest_date)
print("Prepared Zillow shape:", zillow.shape)

zillow.head()

Latest Zillow date used: 2026-05-31
Prepared Zillow shape: (2247, 3)


,neighbourhood,latest_home_value,neighbourhood_clean
4,South Los Angeles,6.725631e+05,south los angeles
8,Southeast Los Angeles,5.999522e+05,southeast los angeles
21,Hollywood,8.973074e+05,hollywood
25,Roosevelt,3.404544e+05,roosevelt
33,East San Jose,1.050129e+06,east san jose


In [ ]:
# Clean Airbnb listing neighborhood

airbnb["neighbourhood_clean"] = (
    airbnb["neighbourhood_cleansed"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Clean Zillow neighborhood

zillow["neighbourhood_clean"] = (
    zillow["neighbourhood"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# First exact merge

merged = airbnb.merge(
    zillow[["neighbourhood_clean", "latest_home_value"]],
    on="neighbourhood_clean",
    how="left"
)

print("Exact merge missing Zillow values:")
print(merged["latest_home_value"].isna().mean() * 100)


Exact merge missing Zillow values:
34.65782886154497


In [ ]:
# Inspect unmatched neighborhoods

unmatched = (
    merged[merged["latest_home_value"].isna()]
    ["neighbourhood_cleansed"]
    .value_counts()
    .head(50)
)

print(unmatched)

neighbourhood_cleansed
Long Beach                               1483
Santa Monica                             1019
West Hollywood                            966
Beverly Hills                             778
Pasadena                                  631
Alhambra                                  622
Exposition Park                           617
Glendale                                  598
Culver City                               529
Hollywood Hills West                      503
Beverly Grove                             478
Rowland Heights                           423
Mid-Wilshire                              381
Inglewood                                 379
Manhattan Beach                           357
Marina del Rey                            352
Redondo Beach                             335
Lancaster                                 330
East Los Angeles                          329
Topanga                                   311
El Monte                                  283
Malibu     

Exact matching was performed first, followed by partial, city-level, and fuzzy matching to account for differences in neighborhood naming conventions between the Airbnb and Zillow datasets and increase the proportion of successfully matched observations.

In [ ]:
# Drop old Zillow columns if they exist
old_cols = [
    "zillow_home_value",
    "zillow_home_value_imputed",
    "zillow_matched_name",
    "zillow_match_type",
    "airbnb_neighborhood_clean"
]
airbnb = airbnb.drop(columns=[c for c in old_cols if c in airbnb.columns])

# Prepare Zillow

zillow = zillow.copy()

# Use latest home value if already prepared
if "latest_home_value" not in zillow.columns:
    date_cols = [col for col in zillow.columns if str(col)[:4].isdigit()]
    latest_date = sorted(date_cols)[-1]
    zillow["latest_home_value"] = pd.to_numeric(zillow[latest_date], errors="coerce")
else:
    latest_date = "latest_home_value"
    zillow["latest_home_value"] = pd.to_numeric(zillow["latest_home_value"], errors="coerce")

# Neighborhood column
if "RegionName" in zillow.columns:
    zillow["zillow_neighborhood"] = zillow["RegionName"]
elif "neighbourhood" in zillow.columns:
    zillow["zillow_neighborhood"] = zillow["neighbourhood"]
else:
    raise ValueError("No Zillow neighborhood column found.")

zillow["zillow_neighborhood_clean"] = (
    zillow["zillow_neighborhood"].astype(str).str.lower().str.strip()
)

# City column
if "City" in zillow.columns:
    zillow["zillow_city_clean"] = (
        zillow["City"].astype(str).str.lower().str.strip()
    )
else:
    zillow["zillow_city_clean"] = np.nan

zillow = zillow.dropna(subset=["latest_home_value"])

# Lookups
neigh_lookup = zillow.groupby("zillow_neighborhood_clean")["latest_home_value"].mean().to_dict()
city_lookup = zillow.dropna(subset=["zillow_city_clean"]).groupby("zillow_city_clean")["latest_home_value"].mean().to_dict()

zillow_neigh_names = list(neigh_lookup.keys())
zillow_city_names = list(city_lookup.keys())

# Prepare Airbnb

airbnb["airbnb_neighborhood_clean"] = (
    airbnb["neighbourhood_cleansed"].astype(str).str.lower().str.strip()
)

unique_places = airbnb["airbnb_neighborhood_clean"].dropna().unique()

# Match function

def match_place(place):
    if place in ["nan", "", "none"]:
        return pd.Series([np.nan, np.nan, "unmatched"])

    # 1. Exact neighborhood
    if place in neigh_lookup:
        return pd.Series([neigh_lookup[place], place, "exact_neighborhood"])

    # 2. Exact city
    if place in city_lookup:
        return pd.Series([city_lookup[place], place, "exact_city"])

    # 3. Zillow neighborhood contains Airbnb name
    contains = zillow[
        zillow["zillow_neighborhood_clean"].str.contains(
            place, case=False, na=False, regex=False
        )
    ]
    if len(contains) > 0:
        return pd.Series([
            contains["latest_home_value"].mean(),
            ", ".join(contains["zillow_neighborhood_clean"].head(3).tolist()),
            "zillow_contains_airbnb"
        ])

    # 4. Airbnb name contains Zillow neighborhood
    contained_by = [
        zname for zname in zillow_neigh_names
        if zname in place and len(zname) >= 5
    ]
    if len(contained_by) > 0:
        vals = [neigh_lookup[zname] for zname in contained_by]
        return pd.Series([
            np.mean(vals),
            ", ".join(contained_by[:3]),
            "airbnb_contains_zillow"
        ])

    # 5. Fuzzy neighborhood
    fuzzy_neigh = process.extractOne(
        place,
        zillow_neigh_names,
        scorer=fuzz.token_sort_ratio
    )

    if fuzzy_neigh and fuzzy_neigh[1] >= 88:
        matched_name = fuzzy_neigh[0]
        return pd.Series([
            neigh_lookup[matched_name],
            matched_name,
            f"fuzzy_neighborhood_{round(fuzzy_neigh[1], 1)}"
        ])

    # 6. Fuzzy city
    fuzzy_city = process.extractOne(
        place,
        zillow_city_names,
        scorer=fuzz.token_sort_ratio
    ) if len(zillow_city_names) > 0 else None

    if fuzzy_city and fuzzy_city[1] >= 90:
        matched_city = fuzzy_city[0]
        return pd.Series([
            city_lookup[matched_city],
            matched_city,
            f"fuzzy_city_{round(fuzzy_city[1], 1)}"
        ])

    return pd.Series([np.nan, np.nan, "unmatched"])

# Match unique neighborhoods only

matches = pd.DataFrame({"airbnb_neighborhood_clean": unique_places})

matches[["zillow_home_value", "zillow_matched_name", "zillow_match_type"]] = (
    matches["airbnb_neighborhood_clean"].apply(match_place)
)

# Merge matches back onto Airbnb
airbnb = airbnb.merge(matches, on="airbnb_neighborhood_clean", how="left")

# Impute remaining missing values
median_home_value = airbnb["zillow_home_value"].median()
airbnb["zillow_home_value_imputed"] = airbnb["zillow_home_value"].fillna(median_home_value)

# Summary
print("Latest Zillow date used:", latest_date)
print("Unique Airbnb neighborhoods:", len(unique_places))
print("Airbnb rows:", airbnb.shape[0])
print("Zillow rows used:", zillow.shape[0])
print()
print("Missing before imputation:", airbnb["zillow_home_value"].isna().sum())
print("Percent missing before imputation:", round(airbnb["zillow_home_value"].isna().mean() * 100, 2))
print("Missing after imputation:", airbnb["zillow_home_value_imputed"].isna().sum())
print("Percent missing after imputation:", round(airbnb["zillow_home_value_imputed"].isna().mean() * 100, 2))
print()
print("Match type counts:")
print(airbnb["zillow_match_type"].value_counts(dropna=False))

airbnb[
    [
        "neighbourhood_cleansed",
        "zillow_home_value",
        "zillow_home_value_imputed",
        "zillow_matched_name",
        "zillow_match_type"
    ]
].head(30)

Latest Zillow date used: latest_home_value
Unique Airbnb neighborhoods: 264
Airbnb rows: 35870
Zillow rows used: 2247

Missing before imputation: 8483
Percent missing before imputation: 23.65
Missing after imputation: 0
Percent missing after imputation: 0.0

Match type counts:
zillow_match_type
exact_neighborhood          16210
unmatched                    8483
zillow_contains_airbnb       7980
airbnb_contains_zillow       2490
fuzzy_neighborhood_91.7       663
fuzzy_neighborhood_90.0        27
fuzzy_neighborhood_100.0       17
Name: count, dtype: int64


,neighbourhood_cleansed,zillow_home_value,zillow_home_value_imputed,zillow_matched_name,zillow_match_type
0,Hollywood,8.973074e+05,8.973074e+05,hollywood,exact_neighborhood
1,Santa Monica,1.895851e+06,1.895851e+06,santa monica pier area/ocean avenue,zillow_contains_airbnb
2,Hollywood,8.973074e+05,8.973074e+05,hollywood,exact_neighborhood
3,Bellflower,NaN,1.037584e+06,NaN,unmatched
4,Pico-Robertson,1.412026e+06,1.412026e+06,pico-robertson,exact_neighborhood
5,Venice,1.848346e+06,1.848346e+06,venice,exact_neighborhood
6,Sherman Oaks,1.362275e+06,1.362275e+06,sherman oaks,exact_neighborhood
7,Koreatown,7.698865e+05,7.698865e+05,koreatown,exact_neighborhood
8,Echo Park,1.112137e+06,1.112137e+06,echo park,exact_neighborhood
9,Carson,5.876896e+05,5.876896e+05,merit-carson,zillow_contains_airbnb


After matching, Zillow home values were available for 27,387 listings (76.35%). Median imputation was used for the remaining unmatched observations to ensure a complete feature set for model training.

In [ ]:
# Final modeling dataset + train/validation/test split + preprocessing

# Select model input features

numeric_features = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "zillow_home_value_imputed",
    "bedrooms",
    "bathrooms"
]

categorical_features = [
    "neighbourhood_cleansed",
    "room_type",
    "property_type"
]

target = "estimated_annual_revenue"

# Createing final cleaned modeling dataset

model_data = airbnb[numeric_features + categorical_features + ["estimated_annual_revenue"]].copy()

# Drop rows missing the target
model_data = model_data.dropna(subset=["estimated_annual_revenue"])

print("Final cleaned modeling dataset shape:", model_data.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

#Saving the final model data as csv
model_data.to_csv(
    "/Users/kalafejzo/Desktop/cleaned_airbnb_zillow_model_data.csv",
    index=False
)


# Splitting into X and y

X = model_data.drop(columns=["estimated_annual_revenue"])
y = model_data["estimated_annual_revenue"]

# 70% training, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Training examples:", X_train.shape[0])
print("Validation examples:", X_val.shape[0])
print("Test examples:", X_test.shape[0])

# Print  numbers

print("\nFINAL NUMBERS")
print("Raw Airbnb shape:", pd.read_csv(airbnb_path).shape)
print("Raw Zillow shape:", pd.read_csv(zillow_path).shape)
print("Final cleaned dataset shape:", model_data.shape)
print("Training examples:", X_train.shape[0])
print("Validation examples:", X_val.shape[0])
print("Test examples:", X_test.shape[0])
print("Target variable:", target)
print("Input features:", numeric_features + categorical_features)

Final cleaned modeling dataset shape: (35870, 12)
Numeric features: ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'availability_365', 'zillow_home_value_imputed', 'bedrooms', 'bathrooms']
Categorical features: ['neighbourhood_cleansed', 'room_type', 'property_type']
Training examples: 25109
Validation examples: 5380
Test examples: 5381

FINAL NUMBERS
Raw Airbnb shape: (45886, 79)
Raw Zillow shape: (21600, 326)
Final cleaned dataset shape: (35870, 12)
Training examples: 25109
Validation examples: 5380
Test examples: 5381
Target variable: estimated_annual_revenue
Input features: ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'availability_365', 'zillow_home_value_imputed', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'room_type', 'property_type']


In [ ]:
# Numerical preprocessing:
# fill missing values with median, then scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# fill missing values with most common category, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Apply correct preprocessing to each feature type
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)